### least square regression optimization using GD & SGD

In [1]:
import numpy as np

## synthetic  data

features_len = 5
samples_len = 1000

true_weights = np.arange(1,1+features_len)

points = []
for i in range(samples_len):
    x = np.random.randn(features_len)
    y = x.dot(true_weights) + np.random.randn()
    points.append((x,y))

In [2]:
points[0]

(array([ 1.61750693,  1.0506155 ,  1.08334584, -0.83120432, -1.13443566]),
 np.float64(-1.4145456649813564))

In [3]:
def least_square_error(w):

    return  sum([(w.dot(x)-y)**2 for x,y in points]) / samples_len

def least_square_error_gradient(w):

    return  sum([ 2*(w.dot(x)-y)*x  for x,y in points ]) / samples_len


def least_square_error_sgd(w,i):

    X,y = points[i]
    
    return ( w.dot(X) - y )**2


def least_square_error_gradient_sgd(w,i):

    X,y = points[i]
    
    return 2 * (w.dot(X) - y) * X



In [4]:
import time

def track_time(fn):

    def wrapper(*args,**kwargs):

        start_time = time.time()

        results = fn(*args,**kwargs)
        
        end_time = time.time()-start_time

        print('Total Time Taken : ',end_time)

        return results

    return wrapper



@track_time
def gradient_descent(n, least_square_fn, least_square_grad):

    w = np.zeros(features_len)
    eta = 0.001
    patience = 0

    best_loss = float('inf')
    loss = 0
    grad = 0

    for i in range(n):

        loss = least_square_fn(w)
        grad = least_square_grad(w)

        w -= eta*grad

        if loss < best_loss:
            best_loss = loss

        else:
            patience +=1

        if patience == 10:
            break

        # print(f'Iteration : [{i+1}/{n}] : Weights : {w} | Loss : {loss} ')
    
    print('-------- Summary --------')
    print('No of epochs taken to converge : ', i)
    print('Final loss : ', loss)
    print('Final weights : ',w)
    print('---------------------------')

    return w, loss

final_weights, final_loss = gradient_descent(10000,least_square_error,least_square_error_gradient)


-------- Summary --------
No of epochs taken to converge :  9488
Final loss :  1.0078248684300892
Final weights :  [1.04186291 1.98279551 2.96141685 4.00904943 5.00495292]
---------------------------
Total Time Taken :  13.343648433685303


In [9]:
## Stochastic Gradient Descent 

    
@track_time
def stochastic_gradient_descent(n, least_square_fn, least_square_grad):

    w = np.zeros(features_len)
    eta = 1
    patience = 0

    best_loss = float('inf')
    loss = 0
    grad = 0
    num_updates = 1.0

    m = np.random.randint(len(points)//2)

    for i in range(n):

        for j in range(m):
            loss = least_square_fn(w,j)
            grad = least_square_grad(w,j)
            w -= eta*grad
            
            eta = 1.0 / num_updates
            num_updates +=1


            if loss < best_loss:
                best_loss = loss

            else:
                patience +=1

            if patience == 10:
                break

        # print(f'Iteration : [{i+1}/{n}] : Weights : {w} | Loss : {loss} ')
    
    print('-------- Summary --------')
    print('No of epochs taken to converge : ', i)
    print('Final loss : ', loss)
    print('Final weights : ',w)
    print('---------------------------')

    return w, loss

final_weights, final_loss = stochastic_gradient_descent(10000,least_square_error_sgd,least_square_error_gradient_sgd)


-------- Summary --------
No of epochs taken to converge :  9999
Final loss :  0.06761015794166077
Final weights :  [1.02556522 2.04588865 2.89128288 4.00833739 4.99542162]
---------------------------
Total Time Taken :  8.735001564025879


In [6]:
# SGD is faster than GD.